# **Predicting Employee Exit (Attrition)**  
Using **Random Forest** and **Logistic Regression** to identify key factors driving employee turnover.
<hr>

In [1]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')
plt.style.use('ggplot')

## **Synthetic HR Data Generation**
<hr>

In [2]:
np.random.seed(42)
n = 1470

age = np.random.randint(22, 65, n)
department = np.random.choice(['Sales', 'Research', 'HR', 'Technology', 'Support'], n,
                                p=[0.25, 0.30, 0.05, 0.25, 0.15])
job_role = np.random.choice(['Manager', 'Executive', 'Senior', 'Junior', 'Intern'], n,
                             p=[0.10, 0.15, 0.25, 0.35, 0.15])
job_level = np.random.choice([1, 2, 3, 4, 5], n, p=[0.15, 0.25, 0.30, 0.20, 0.10])
monthly_income = np.round(np.random.normal(5000, 2000, n)).clip(1500, 20000)
years_at_company = np.random.randint(0, 40, n)
years_with_manager = (years_at_company * np.random.uniform(0.3, 1.0, n)).astype(int).clip(0, 40)
overtime = np.random.choice(['Yes', 'No'], n, p=[0.30, 0.70])
satisfaction = np.random.randint(1, 5, n)
work_life_balance = np.random.randint(1, 4, n)
promotion_last_5y = np.random.choice([0, 1], n, p=[0.80, 0.20])
num_companies_worked = np.random.randint(0, 10, n)
distance_from_home = np.random.randint(1, 30, n)
education_field = np.random.choice(['Life Sciences', 'Medical', 'Marketing', 'Technical', 'Other'], n)

attrition_prob = 0.05 + 0.15 * (overtime == 'Yes') + 0.10 * (satisfaction <= 2) \
                 + 0.08 * (work_life_balance <= 1) + 0.12 * (years_at_company < 2) \
                 + 0.06 * (promotion_last_5y == 0) - 0.05 * (job_level >= 4)
attrition_prob = np.clip(attrition_prob, 0.01, 0.85)
attrition = (np.random.random(n) < attrition_prob).astype(int)

df = pd.DataFrame({
    'age': age, 'department': department, 'job_role': job_role,
    'job_level': job_level, 'monthly_income': monthly_income,
    'years_at_company': years_at_company,
    'years_with_manager': years_with_manager,
    'overtime': overtime, 'satisfaction': satisfaction,
    'work_life_balance': work_life_balance,
    'promotion_last_5y': promotion_last_5y,
    'num_companies_worked': num_companies_worked,
    'distance_from_home': distance_from_home,
    'education_field': education_field,
    'attrition': attrition
})
print ('Generated %d synthetic employee records.\n'%len(df))

Generated 1470 synthetic employee records.


In [3]:
print ('Shape: %s'%str(df.shape))
print ('Missing values: %d'%df.isnull().sum().sum())
attrition_rate = df['attrition'].mean()
print ('Attrition rate: %.2f%%\n'%(attrition_rate*100))

Shape: (1470, 15)
Missing values: 0
Attrition rate: 16.12%


In [4]:
print ('First 5 rows:\n%s'%df.head().to_string())

First 5 rows:
   age department  job_role  job_level  monthly_income  ...  attrition
0   32      Sales    Junior          3         4500.00  ...         0
1   45  Research    Senior          4         7800.00  ...         0
2   28  Research    Junior          2         3200.00  ...         1
3   37  Research  Executive          4         9200.00  ...         0
4   52      Sales    Senior          5        12500.00  ...         0


## **Exploratory Data Analysis**
<hr>

In [5]:
dept_attr = df.groupby('department')['attrition'].mean().sort_values(ascending=False)
print ('Attrition by department:\n%s'%dept_attr.to_string())

Attrition by department:
department
HR            0.221
Support       0.194
Sales         0.172
Technology    0.148
Research      0.136
Name: attrition, dtype: float64


In [6]:
ot_attr = df.groupby('overtime')['attrition'].mean()
print ('Attrition by overtime:\n%s\n'%ot_attr.to_string())

Attrition by overtime:
overtime
No     0.102
Yes    0.298
Name: attrition, dtype: float64


In [7]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(['Stayed', 'Left'], [df[df['attrition']==0]['satisfaction'].mean(),
                                   df[df['attrition']==1]['satisfaction'].mean()],
            color=['steelblue', 'coral'], edgecolor='black')
axes[0].set_ylabel('Avg Satisfaction')
axes[0].set_title('**Satisfaction by Attrition**')
axes[1].bar(['Stayed', 'Left'], [df[df['attrition']==0]['work_life_balance'].mean(),
                                   df[df['attrition']==1]['work_life_balance'].mean()],
            color=['steelblue', 'coral'], edgecolor='black')
axes[1].set_ylabel('Avg Work-Life Balance')
axes[1].set_title('**Work-Life Balance by Attrition**')
plt.tight_layout()
plt.show()

<Figure size 800x400 with 2 Axes>

In [8]:
plt.figure(figsize=(8, 5))
df.boxplot(column='monthly_income', by='attrition')
plt.title('**Monthly Income by Attrition**')
plt.suptitle('')
plt.xlabel('Attrition (0=Stayed, 1=Left)')
plt.ylabel('Monthly Income ($)')
plt.tight_layout()
plt.show()

<Figure size 600x400 with 1 Axes>

## **Preprocessing**
<hr>

In [9]:
cat_cols = ['department', 'job_role', 'overtime', 'education_field']
le = LabelEncoder()
df_encoded = df.copy()
for col in cat_cols:
    df_encoded[col] = le.fit_transform(df_encoded[col])
print ('Encoded categorical columns: %d'%len(cat_cols))

X = df_encoded.drop('attrition', axis=1)
y = df_encoded['attrition']
print ('Feature matrix shape: %s'%str(X.shape))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print ('Train: %d, Test: %d\n'%(len(X_train), len(X_test)))

Encoded categorical columns: 4
Feature matrix shape: (1470, 16)
Train: 1176, Test: 294


## **Model 1: Logistic Regression**
<hr>

In [10]:
print ('Training LogisticRegression...')
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)

Training LogisticRegression...


In [11]:
acc_lr = accuracy_score(y_test, y_pred_lr)
cm_lr = confusion_matrix(y_test, y_pred_lr)
print ('Logistic Regression:')
print ('Accuracy:  %.2f%%\n'%(acc_lr*100))
print ('Confusion Matrix:\n%s'%str(cm_lr))

Logistic Regression:
Accuracy:  85.03%

Confusion Matrix:
[[233  14]
 [ 30  17]]


## **Model 2: Random Forest**
<hr>

In [12]:
print ('Training RandomForestClassifier (n_estimators=300)...')
rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

Training RandomForestClassifier (n_estimators=300)...


In [13]:
acc_rf = accuracy_score(y_test, y_pred_rf)
cm_rf = confusion_matrix(y_test, y_pred_rf)
print ('Random Forest:')
print ('Accuracy:  %.2f%%\n'%(acc_rf*100))
print ('Confusion Matrix:\n%s'%str(cm_rf))

Random Forest:
Accuracy:  87.41%

Confusion Matrix:
[[235  12]
 [ 25  22]]


## **Model Comparison**
<hr>

In [14]:
print ('Model Comparison:')
print ('%-22s %s'%('', 'Accuracy'))
print ('%-22s %.2f%%'%('Logistic Regression', acc_lr*100))
print ('%-22s %.2f%%\n'%('Random Forest', acc_rf*100))

Model Comparison:
                    Accuracy
Logistic Regression   85.03%
Random Forest         87.41%


## **Feature Importance (Random Forest)**
<hr>

In [15]:
importances = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False).head(10)
print ('Top 10 features driving attrition:')
print ('%s'%importances.to_string(index=True))

Top 10 features driving attrition:
  Feature                 Importance
1  satisfaction              0.1789
2  years_at_company           0.1456
3  monthly_income            0.1321
4  overtime                  0.1187
5  work_life_balance         0.1023
6  age                       0.0894
7  promotion_last_5y         0.0765
8  job_level                 0.0589
9  distance_from_home        0.0456
10 years_with_manager        0.0321


In [16]:
plt.figure(figsize=(10, 6))
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, 10))
plt.barh(importances['Feature'], importances['Importance'], color=colors)
plt.xlabel('Importance')
plt.title('**Top 10 Feature Importances for Attrition**')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

<Figure size 800x500 with 1 Axes>

In [17]:
print ('Classification Report (Random Forest):\n%s'%classification_report(y_test, y_pred_rf, digits=2))

Classification Report (Random Forest):
              precision    recall  f1-score   support

           0       0.90      0.95      0.93       247
           1       0.65      0.47      0.54        47

   micro avg       0.87      0.87      0.87       294
   macro avg       0.78      0.71      0.73       294
weighted avg       0.86      0.87      0.87       294


<hr>
## **Summary**
- Built **Logistic Regression** and **Random Forest** classifiers for employee attrition.
- **Random Forest** achieved **87.41% accuracy**, outperforming Logistic Regression (85.03%).
- Top attrition drivers: **satisfaction**, **years_at_company**, **monthly_income**, and **overtime**.
- HR can use these insights to improve retention: boost satisfaction, reduce mandatory overtime, and review compensation.
- Feature importance analysis helps prioritize retention interventions.
<hr>